In [2]:
import os
import torch
import numpy as np
from extract import extract_graph, extract_die_area, load_file_content
import scipy.sparse as sp 
from helper import normalize_features, build_X_hop_mask , get_compressed_graph , relative_masking , load_cts_parameters
import pandas as pd

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
base_dir = "dataset_with_def/placement_files/"
csv_dir = "dataset_with_def/"

# --- CONTROL KNOB ---
max_designs_to_load = 500# Set this to however many designs you want to test right now
# --------------------

training_dataset = []
global_max_k = 0
global_max_cell_types = 0
loaded_count = 0

print(f"Starting Multi-Graph Pipeline (Targeting {max_designs_to_load} designs)...\n" + "="*50)

# Iterate over every folder in your base directory
for run_folder in os.listdir(base_dir):
    # Stop if we hit our requested limit
    if loaded_count >= max_designs_to_load:
        print(f"\nReached target limit of {max_designs_to_load} designs. Stopping extraction.")
        break

    folder_path = os.path.join(base_dir, run_folder)
    
    if not os.path.isdir(folder_path):
        continue
        
    design_name = run_folder.split("_")[0]
    print(f"\nProcessing Design: {design_name} (Run: {run_folder})")

    if design_name in ['ethmac']:
        clock_port = "wb_clk_i"
    else:
        clock_port = "clk"
    
    # 1. Build Paths
    def_path = os.path.join(folder_path, f"{design_name}.def")
    saif_path = os.path.join(folder_path, f"{design_name}.saif")
    timing_path = os.path.join(folder_path, "timing_paths.csv")
    csv_path = os.path.join(csv_dir, f"unified_manifest_normalized.csv")
    
    if not (os.path.exists(def_path) and os.path.exists(saif_path) and os.path.exists(timing_path)):
        print(f"  -> WARNING: Missing files for {design_name}. Skipping.")
        continue

    # 2. Extract Raw Graph Data
    graph = extract_graph(def_path, saif_path, timing_path, clock_port=clock_port)
    def_text = load_file_content(def_path)
    die_x_min, die_y_min, die_x_max, die_y_max = extract_die_area(def_text)
    
    num_nodes = len(graph['nodes'])
    print(f"  -> Nodes: {num_nodes} | Undirected Edges: {graph['undirected_edges'].shape[1]}")
    
    # 3. Process REAL Features
    X_float, X_cell_ids, norm_stats = normalize_features(
        graph['nodes'], 
        die_x_min, die_y_min, die_x_max, die_y_max
    )

    # 4. Build the 3-Hop Mask (The P matrix structure)
    rows, cols = build_X_hop_mask(num_nodes, graph['undirected_edges'], hop_mask_len=2)
    p_indices = torch.tensor(np.array([rows, cols]), dtype=torch.long)

    # 5. Build the Skip Edges (Timing paths) Sparse Matrix
    if len(graph['skip_edges']) > 0:
        skip_rows = torch.tensor(graph['skip_edges'][:, 0], dtype=torch.long)
        skip_cols = torch.tensor(graph['skip_edges'][:, 1], dtype=torch.long)
    else:
        skip_rows, skip_cols = torch.tensor([], dtype=torch.long), torch.tensor([], dtype=torch.long)
        
    skip_indices = torch.stack([skip_rows, skip_cols])
    skip_vals = torch.ones(skip_rows.size(0))
    A_skip_csr = torch.sparse_coo_tensor(skip_indices, skip_vals, (num_nodes, num_nodes)).coalesce().to_sparse_csr()

    # 6. Calculate K for this specific design
    n_ff = (X_float[:, 10] == 1.0).sum().item()
    current_k = min(int(n_ff//2), 1800)
    current_k = max(current_k,1)
    
    # Update Global Maximums
    global_max_k = max(global_max_k, current_k)
    global_max_cell_types = max(global_max_cell_types, int(X_cell_ids.max().item() + 1))

    # ==========================================
    # 5.5 Build the 1-Hop Wire Edges (FIXED)
    # ==========================================
    if graph['undirected_edges'].shape[0] > 0:
        # Indices are currently [Edges, 2]. We need to split the COLUMNS.
        wire_rows = torch.tensor(graph['undirected_edges'][:, 0], dtype=torch.long)
        wire_cols = torch.tensor(graph['undirected_edges'][:, 1], dtype=torch.long)
    else:
        wire_rows, wire_cols = torch.tensor([], dtype=torch.long), torch.tensor([], dtype=torch.long)
        
    wire_indices = torch.stack([wire_rows, wire_cols])
    wire_vals = torch.ones(wire_rows.size(0))
    
    # This NxN sparse matrix will now correctly have ~36,000 entries
    A_wire_csr = torch.sparse_coo_tensor(
        wire_indices, 
        wire_vals, 
        (num_nodes, num_nodes)
    ).coalesce().to_sparse_csr()


    cts_runs = load_cts_parameters(csv_path, placement_id=run_folder, device=device)
    raw_areas = torch.tensor([[n['cell_area']] for n in graph['nodes']], dtype=torch.float32)



    # 7. Package and Store in memory
    graph_data = {
        'run_folder': run_folder,
        'name': design_name,
        'num_nodes': num_nodes,
        'X': X_float,
        'X_cell_ids': X_cell_ids,
        'p_indices': p_indices,
        'A_skip_csr': A_skip_csr,
        'current_k': current_k, 
        'A_wire_csr': A_wire_csr,
        'cts_runs': cts_runs, 
        'raw_areas': raw_areas
    }
    training_dataset.append(graph_data)
    loaded_count += 1
    torch.save(graph_data, os.path.join("processed_graphs/", f"{run_folder}.pt"))
    print(f"  -> Saved {run_folder}.pt to disk.")
    print(f"  -> Successfully packaged! Required K: {current_k}")

print("\n" + "="*50)
print(f"Total Designs Loaded: {len(training_dataset)}")
print(f"GLOBAL MAX CELL TYPES: {global_max_cell_types}")


torch.save({
    "global_max_k": global_max_k,
    "global_max_cell_types": global_max_cell_types
}, "global_metadata.pt")



Starting Multi-Graph Pipeline (Targeting 500 designs)...

Processing Design: picorv32 (Run: picorv32_run_20260306_135024)
Flip-flops: 1597
Unique pairs: 3158, dropped: 0
Skip edges shape: (3158, 2)
Nodes: 6801
Directed edges: 18072
Undirected edges: 36144
Flip-flops: 1597
Fan-in  — max: 6, avg: 2.7
Fan-out — max: 139, avg: 2.7
  -> Nodes: 6801 | Undirected Edges: 2
  -> Hop 2 expansion complete.
Mask created! 332,856 total connections allowed.
  -> Saved picorv32_run_20260306_135024.pt to disk.
  -> Successfully packaged! Required K: 798

Processing Design: aes (Run: aes_run_20260307_110223)
Flip-flops: 2994
Unique pairs: 5596, dropped: 0
Skip edges shape: (5596, 2)
Nodes: 15962
Directed edges: 45797
Undirected edges: 91594
Flip-flops: 2994
Fan-in  — max: 6, avg: 2.9
Fan-out — max: 172, avg: 2.9
  -> Nodes: 15962 | Undirected Edges: 2
  -> Hop 2 expansion complete.
Mask created! 1,251,526 total connections allowed.
  -> Saved aes_run_20260307_110223.pt to disk.
  -> Successfully packag

KeyboardInterrupt: 

In [2]:
#USE THIS IF YOU CHANGED NUMBER OF HOPS

import os
import torch

processed_dir = "processed_graphs"
files = [f for f in os.listdir(processed_dir) if f.endswith('.pt')]

true_max_cell_id = 0
true_max_k = 0

print("🔍 Scanning processed files to repair metadata...")
for f in files:
    data = torch.load(os.path.join(processed_dir, f))
    
    # Find the actual max cell ID used in this file
    file_max_cell = int(data['X_cell_ids'].max().item())
    true_max_cell_id = max(true_max_cell_id, file_max_cell)
    
    # Find the actual max K used in this file
    file_max_k = int(data['current_k'])
    true_max_k = max(true_max_k, file_max_k)

# Add +1 because indices are 0-based (if max ID is 425, you need 426 slots)
final_cell_types = true_max_cell_id + 1

torch.save({
    "global_max_k": true_max_k,
    "global_max_cell_types": final_cell_types
}, "global_metadata.pt")

print(f"✅ Metadata Repaired!")
print(f"   -> New num_cell_types: {final_cell_types}")
print(f"   -> New max_k: {true_max_k}")

🔍 Scanning processed files to repair metadata...
✅ Metadata Repaired!
   -> New num_cell_types: 425
   -> New max_k: 1800


In [1]:
# ==========================================
# FULL FEATURE X-RAY FOR SUPERNODE 278
# ==========================================
target_cluster = 278
print(f"\n🔍 FULL FEATURE X-RAY: SUPERNODE {target_cluster}")
print("="*50)

# Get the features for just this supernode
sn_features = X_tilde[target_cluster].cpu().numpy()

# Note: Adjust these names based on your actual extraction order!
feature_names = [
    "0: X Coordinate (Avg)", "1: Y Coordinate (Avg)", 
    "2: Base Feature 3", "3: Base Feature 4", "4: Base Feature 5", 
    "5: Base Feature 6", "6: Cell Area (Sum)", "7: Base Feature 8",
    "8: Base Feature 9", "9: Base Feature 10", "10: Is_FF (Boolean)", 
    # ... (Fill in the rest up to 18 based on your raw X matrix)
]

print("--- Base Features (from C_norm.t() @ X) ---")
for i in range(18): # Assuming first 18 are from X_tilde_base
    name = feature_names[i] if i < len(feature_names) else f"{i}: Unknown Feature"
    print(f"  Feature {name:<25}: {sn_features[i]:>8.4f}")

print("\n--- Phase 2 Added Features ---")
# If you appended Count and Spread at the end:
print(f"  Feature 18: Total Area/Count     : {sn_features[-2]:>8.4f}")
print(f"  Feature 19: Physical Spread (Var): {sn_features[-1]:>8.4f}")


🔍 FULL FEATURE X-RAY: SUPERNODE 278


NameError: name 'X_tilde' is not defined

In [ ]:
data = torch.load("processed_graphs/picorv32_run_20260306_135024.pt")
print(data['cts_runs'][0]['targets']['wl'].item())
# Should be ~ -1.197
# If it's ~ 500000, regenerate all .pt files

-1.1974513530731201


In [1]:
#for test set

import os
import torch
import numpy as np
from extract import extract_graph, extract_die_area, load_file_content
import scipy.sparse as sp 
from helper import normalize_features, build_X_hop_mask , get_compressed_graph , relative_masking , load_cts_parameters
import pandas as pd

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
base_dir = "dataset_with_def/placement_files/"
csv_dir = "dataset_with_def/"

# --- CONTROL KNOB ---
max_designs_to_load = 1# Set this to however many designs you want to test right now
# --------------------

training_dataset = []
global_max_k = 0
global_max_cell_types = 0
loaded_count = 0

print(f"Starting Multi-Graph Pipeline (Targeting {max_designs_to_load} designs)...\n" + "="*50)

# Iterate over every folder in your base directory
for run_folder in os.listdir(base_dir):
    # Stop if we hit our requested limit
    if loaded_count >= max_designs_to_load:
        print(f"\nReached target limit of {max_designs_to_load} designs. Stopping extraction.")
        break

    folder_path = os.path.join(base_dir, run_folder)
    
    if not os.path.isdir(folder_path):
        continue
        
    design_name = run_folder.split("_")[0]

    if design_name != "zipdiv":
        continue

    print(f"\nProcessing Design: {design_name} (Run: {run_folder})")

    if design_name in ['ethmac']:
        clock_port = "wb_clk_i"
    elif design_name in ['zipdiv']:
        clock_port = "i_clk"
    else:
        clock_port = "clk"
    
    # 1. Build Paths
    def_path = os.path.join(folder_path, f"{design_name}.def")
    saif_path = os.path.join(folder_path, f"{design_name}.saif")
    timing_path = os.path.join(folder_path, "timing_paths.csv")
    csv_path = os.path.join(csv_dir, f"experiment_log.csv")
    
    if not (os.path.exists(def_path) and os.path.exists(saif_path) and os.path.exists(timing_path)):
        print(f"  -> WARNING: Missing files for {design_name}. Skipping.")
        continue

    # 2. Extract Raw Graph Data
    graph = extract_graph(def_path, saif_path, timing_path, clock_port=clock_port)
    def_text = load_file_content(def_path)
    die_x_min, die_y_min, die_x_max, die_y_max = extract_die_area(def_text)
    
    num_nodes = len(graph['nodes'])
    print(f"  -> Nodes: {num_nodes} | Undirected Edges: {graph['undirected_edges'].shape[1]}")
    
    # 3. Process REAL Features
    X_float, X_cell_ids, norm_stats = normalize_features(
        graph['nodes'], 
        die_x_min, die_y_min, die_x_max, die_y_max
    )

    # 4. Build the 3-Hop Mask (The P matrix structure)
    rows, cols = build_X_hop_mask(num_nodes, graph['undirected_edges'], hop_mask_len=2)
    p_indices = torch.tensor(np.array([rows, cols]), dtype=torch.long)

    # 5. Build the Skip Edges (Timing paths) Sparse Matrix
    if len(graph['skip_edges']) > 0:
        skip_rows = torch.tensor(graph['skip_edges'][:, 0], dtype=torch.long)
        skip_cols = torch.tensor(graph['skip_edges'][:, 1], dtype=torch.long)
    else:
        skip_rows, skip_cols = torch.tensor([], dtype=torch.long), torch.tensor([], dtype=torch.long)
        
    skip_indices = torch.stack([skip_rows, skip_cols])
    skip_vals = torch.ones(skip_rows.size(0))
    A_skip_csr = torch.sparse_coo_tensor(skip_indices, skip_vals, (num_nodes, num_nodes)).coalesce().to_sparse_csr()

    # 6. Calculate K for this specific design
    n_ff = (X_float[:, 10] == 1.0).sum().item()
    current_k = min(int(n_ff//2), 1800)
    current_k = max(current_k,1)
    
    # Update Global Maximums
    global_max_k = max(global_max_k, current_k)
    global_max_cell_types = max(global_max_cell_types, int(X_cell_ids.max().item() + 1))

    # ==========================================
    # 5.5 Build the 1-Hop Wire Edges (FIXED)
    # ==========================================
    if graph['undirected_edges'].shape[0] > 0:
        # Indices are currently [Edges, 2]. We need to split the COLUMNS.
        wire_rows = torch.tensor(graph['undirected_edges'][:, 0], dtype=torch.long)
        wire_cols = torch.tensor(graph['undirected_edges'][:, 1], dtype=torch.long)
    else:
        wire_rows, wire_cols = torch.tensor([], dtype=torch.long), torch.tensor([], dtype=torch.long)
        
    wire_indices = torch.stack([wire_rows, wire_cols])
    wire_vals = torch.ones(wire_rows.size(0))
    
    # This NxN sparse matrix will now correctly have ~36,000 entries
    A_wire_csr = torch.sparse_coo_tensor(
        wire_indices, 
        wire_vals, 
        (num_nodes, num_nodes)
    ).coalesce().to_sparse_csr()


    # cts_runs = load_cts_parameters(csv_path, placement_id=run_folder, device=device)
    


    # 7. Package and Store in memory
    graph_data = {
        'run_folder': run_folder,
        'name': design_name,
        'num_nodes': num_nodes,
        'X': X_float,
        'X_cell_ids': X_cell_ids,
        'p_indices': p_indices,
        'A_skip_csr': A_skip_csr,
        'current_k': current_k, 
        'A_wire_csr': A_wire_csr,
    
    }
    training_dataset.append(graph_data)
    loaded_count += 1
    torch.save(graph_data, os.path.join("processed_graphs/", f"{run_folder}.pt"))
    print(f"  -> Saved {run_folder}.pt to disk.")
    print(f"  -> Successfully packaged! Required K: {current_k}")

print("\n" + "="*50)
print(f"Total Designs Loaded: {len(training_dataset)}")
print(f"GLOBAL MAX CELL TYPES: {global_max_cell_types}")


torch.save({
    "global_max_k": global_max_k,
    "global_max_cell_types": global_max_cell_types
}, "global_metadata.pt")



Starting Multi-Graph Pipeline (Targeting 1 designs)...

Processing Design: zipdiv (Run: zipdiv_run_20260312_160558)
Building cell vocabulary from sky130_fd_sc_hd_tt_025C_1v80.lib...
Vocabulary: 428 unique cells
Flip-flops: 142
Unique pairs: 281, dropped: 0
Skip edges shape: (281, 2)
Nodes: 1169
Directed edges: 3222
Undirected edges: 6444
Flip-flops: 142
Fan-in  — max: 5, avg: 2.8
Fan-out — max: 111, avg: 2.8
  -> Nodes: 1169 | Undirected Edges: 2
  -> Hop 2 expansion complete.
Mask created! 53,922 total connections allowed.
  -> Saved zipdiv_run_20260312_160558.pt to disk.
  -> Successfully packaged! Required K: 71

Reached target limit of 1 designs. Stopping extraction.

Total Designs Loaded: 1
GLOBAL MAX CELL TYPES: 424


/tmp/ipykernel_41761/2284227086.py:89: UserWarning: Sparse CSR tensor support is in beta state. If you miss a functionality in the sparse tensor support, please submit a feature request to https://github.com/pytorch/pytorch/issues. (Triggered internally at /pytorch/aten/src/ATen/SparseCsrTensorImpl.cpp:49.)
  A_skip_csr = torch.sparse_coo_tensor(skip_indices, skip_vals, (num_nodes, num_nodes)).coalesce().to_sparse_csr()
